# Lab 2: Rotation

In [ ]:
# you'll need at least these packages today
import numpy as np
import matplotlib.pyplot as plt

import lightkurve as lk
from astropy.timeseries import LombScargle
# import batman
from fleck import generate_spots, Star
import astropy.units as u

# Goals:
1. Work with a partner (make friends, do great science!) Write down who your partner is in your notebook!
2. Download the Kepler data (1800 second cadence) for this start with known rotation 
4. Plot the light curve as before (raw, and stiched and flattened)
5. Compute and plot the L-S periodogram and the ACF for the entire flattened light curve. What period(s) do you see recovered? Do they agree?
6. Compute and plot the L-S periodogram and the ACF for a small portion of the light curve (e.g. 90 days). Do you get the same period?
7. How does the width of the L-S periodogram peak change between the two (4-years versus 4-months of data)?
9. Zoom in on a very small portion of the light curve (2X the rotation period). Using `fleck`, try to generate a lightcurve with similar shape. How many spots do you need?( Note: you'll probably need to normalize the simulated light curves to get them to line up with the data.)
13. Rename your notebook file to YOUR name(s), turn in via [Dropbox upload link](https://www.dropbox.com/request/6szx9GBOVKs5wK4F37K5) 

In [ ]:
LC = lk.search_lightcurve('KIC 8382212', author='Kepler').download_all()


In [ ]:
LC.stitch().plot()
plt.show()


In [ ]:

def ACF(y):
    ''' 
    Computes the autocorrelation of a 1D signal (e.g. flux)
    Assumes data is uniformly sampled
    
    Returns same size array (i.e. computes ACF for every possible shift)
    '''
    data = y - np.nanmean(y) # center data around 0
    result = np.correlate(data, data, mode='full')
    
    # only give half the correlation, since its symmetric
    return result[result.size // 2:] 



In [ ]:
spot_contrast = 0.7 # typical contrast for spots
u_ld = [0.5079, 0.2239] # quadratic limb darkening

incl = 77*u.deg # stellar inclination (0deg = pole on, 90deg = equator on)

# place spots on surface
spot_rads = np.array([[0.2], [0.3]]) # units of r_spot/r_star
spot_lons = np.array([[122], [275]]) * u.deg # longitude (0,360)
spot_lats = np.array([[25], [45]]) * u.deg # latitude (-90,90)


# create an array of times (could change for actual times of data)
times = np.linspace(0, 100, 1000)

# generate a rotating star
star = Star(spot_contrast=spot_contrast, n_phases=len(times), u_ld=u_ld, 
            rotation_period=11)

# render the light curve
lcs = star.light_curve(spot_lons, spot_lats, spot_rads, incl, times=times, 
                       time_ref=0 # time when lon=0 faces observer
                      )

# plot simulated light curve
plt.plot(times, lcs.flatten())
plt.xlabel('time (days)')
plt.ylabel('Relative Flux')
plt.show()

# make cartoon of star
star.plot(spot_lons, spot_lats, spot_rads, incl, 
          time=5.1, 
          time_ref=0)
plt.show()

# demo how to use ACF
model_ACF = ACF(lcs.flatten())
plt.plot(model_ACF)
plt.xlabel('lag (days')
plt.ylabel('ACF')
plt.show()

## some helpful links
- https://fleck.readthedocs.io/en/stable/fleck/gettingstarted.html
- https://docs.astropy.org/en/stable/timeseries/lombscargle.html
- https://ipython-books.github.io/103-computing-the-autocorrelation-of-a-time-series/
- https://ui.adsabs.harvard.edu/abs/2014ApJS..211...24M/abstract